In [1]:
pip install xgboost hmmlearn yfinance optuna scikit-learn pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ==============================================================================
#  MODEL C — STOCK-LEVEL CRASH RISK DETECTOR
#  Binary Classifier: Will this stock drop >10% in the next 10 days?
#  Model: XGBoost (drop-in replacement for LightGBM, works on Python 3.13)
# ==============================================================================
 
# ── BLOCK 1: IMPORTS ──────────────────────────────────────────────────────────
# We bring in every tool we need before writing any logic.
# Think of imports like gathering all your ingredients before you start cooking.
 
import numpy as np
# numpy  → the foundation of all numerical work in Python.
#          Gives us arrays, math operations, and statistical functions.
#          We use it for things like .reshape(), .min(), and array arithmetic.
 
import pandas as pd
# pandas → the spreadsheet of Python.
#          Everything lives in a DataFrame (rows × columns).
#          We use it to store prices, returns, features, and labels together.
 
import yfinance as yf
# yfinance → pulls historical stock price data from Yahoo Finance for free.
#            We just pass a ticker symbol and date range — it handles the rest.
 
import xgboost as xgb
# xgboost → our machine learning model.
#           XGBoost builds hundreds of small decision trees, each one learning
#           from the mistakes of the previous one. This process is called
#           "gradient boosting." It's one of the most battle-tested algorithms
#           in quantitative finance.
 
from hmmlearn import hmm
# hmmlearn → Hidden Markov Model library.
#            An HMM finds hidden "states" (Bull / Bear / Sideways) by looking
#            at observable data (daily returns). We use this to capture the
#            macro market environment — one stock's crash risk is higher
#            when the entire market is in a Bear regime.
 
from sklearn.preprocessing import StandardScaler
# StandardScaler → rescales all features to have mean=0 and std=1.
#                  Why? XGBoost can handle raw numbers, but scaling makes
#                  training more stable and speeds up convergence.
#                  Formula: scaled_value = (value - mean) / std
 
from sklearn.metrics import (
    classification_report,
    # classification_report → prints a table of Precision, Recall, F1-Score
    #                         for each class (Crash vs No-Crash).
 
    confusion_matrix,
    # confusion_matrix → 2×2 grid showing True Positives, False Positives,
    #                    True Negatives, False Negatives.
 
    roc_auc_score
    # roc_auc_score → Area Under the ROC Curve.
    #                 Measures how well the model RANKS crash days above
    #                 safe days, regardless of the threshold chosen.
    #                 0.5 = random guessing. 1.0 = perfect.
)
 
import optuna
# optuna → Bayesian hyperparameter optimiser.
#          Instead of manually guessing good settings for XGBoost,
#          Optuna runs multiple trials and intelligently narrows in
#          on the best combination of hyperparameters.
 
import warnings
warnings.filterwarnings('ignore')
# warnings → we silence non-critical library warnings so the output stays clean.
#            This does NOT hide real errors — only deprecation notices etc.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ── BLOCK 2: CONFIGURATION ────────────────────────────────────────────────────
# All tunable settings live at the top in one place.
# If you want to change dates, tickers, or thresholds — change them HERE only.
START_DATE      = "2008-01-01"
END_DATE        = "2024-12-31"
TRAIN_CUTOFF    = "2022-06-30"
TEST_START      = "2022-07-01"
CRASH_WINDOW    = 10
CRASH_THRESHOLD = 0.10
N_HMM_STATES   = 3
OPTUNA_TRIALS   = 30

TICKERS = [
    "RELIANCE.NS",   "TCS.NS",        "HDFCBANK.NS",  "INFY.NS",
    "ICICIBANK.NS",  "HINDUNILVR.NS", "ITC.NS",        "SBIN.NS",
    "BHARTIARTL.NS", "KOTAKBANK.NS"
]

In [4]:
# ── BLOCK 3: DOWNLOAD PRICE DATA ──────────────────────────────────────────────
 
print("=" * 60)
print("STEP 1: DOWNLOADING PRICE DATA")
print("=" * 60)
 
raw = yf.download(
    TICKERS,          # Which stocks to download
    start=START_DATE, # From this date
    end=END_DATE,     # To this date
    auto_adjust=True  # Automatically adjust for splits and dividends
)
# yf.download() → contacts Yahoo Finance and returns a multi-level DataFrame.
#                 It contains Open, High, Low, Close, Volume for every ticker.
#                 auto_adjust=True ensures prices are "clean" — a 2-for-1
#                 stock split won't look like a -50% crash in our data.
 
prices = raw["Close"].dropna(how="all")
# raw["Close"]      → selects only the Closing Price column from the multi-level df.
# .dropna(how="all")→ removes any row where EVERY stock has a NaN.
#                     (e.g., weekends, holidays). Keeps rows where at least
#                     one stock has a valid price.
 
print(f"Downloaded {prices.shape[0]} trading days × {prices.shape[1]} stocks\n")
# prices.shape → (rows, columns). Typical output: (1762, 10)
# Add this check at the end of Block 3
print(f"Tickers loaded: {prices.shape[1]}")
print(f"Missing: {[t for t in TICKERS if t not in prices.columns]}")

STEP 1: DOWNLOADING PRICE DATA


[*********************100%***********************]  10 of 10 completed

Downloaded 4188 trading days × 10 stocks

Tickers loaded: 10
Missing: []


In [5]:
# ── BLOCK 4: COMPUTE DAILY RETURNS ────────────────────────────────────────────
 
returns = prices.pct_change().fillna(0)
# pct_change() → computes: (today - yesterday) / yesterday for every cell.
#                This converts raw prices into daily percentage returns.
#                Example: price goes from 100 to 103 → return = 0.03 (3%).
# .fillna(0)   → the very first row has no "yesterday", so it becomes NaN.
#                We fill it with 0 (flat return) rather than dropping the row.

In [6]:
# ── BLOCK 5: TRAIN HMM (MACRO REGIME DETECTOR) ───────────────────────────────
 
print("=" * 60)
print("STEP 2: TRAINING HIDDEN MARKOV MODEL")
print("=" * 60)
 
nifty_raw = yf.download("^NSEI", start=START_DATE, end=END_DATE, auto_adjust=True)
# ^NSEI → Yahoo Finance ticker for the NIFTY 50 index.
#         We use the index (not individual stocks) because the HMM needs
#         to learn the OVERALL MARKET regime, not a single stock's behavior.
 
nifty_ret = nifty_raw["Close"].pct_change().fillna(0)
# Same as before: convert NIFTY prices into daily returns.
 
nifty_values = nifty_ret.values.reshape(-1, 1)
# .values     → converts the pandas Series into a plain numpy array.
# .reshape(-1, 1) → HMM requires input shape (n_samples, n_features).
#                   -1 means "figure out this dimension automatically."
#                   So a 1762-element array becomes shape (1762, 1).
 
hmm_model = hmm.GaussianHMM(
    n_components=N_HMM_STATES,  # 3 hidden states
    covariance_type="full",     # Each state has its own full covariance matrix
    n_iter=200,                 # Maximum EM algorithm iterations
    random_state=42             # Seed for reproducibility
)
# hmm.GaussianHMM → a Hidden Markov Model where each hidden state emits
#                   observations drawn from a Gaussian (normal) distribution.
#                   "full" covariance means each state gets its own mean AND
#                   variance — most flexible option.
 
hmm_model.fit(nifty_values)
# .fit() → runs the Expectation-Maximisation (EM) algorithm.
#           It alternates between:
#           E-step: given current parameters, what state is each day in?
#           M-step: given the state assignments, update the parameters.
#           After 200 iterations (or earlier convergence), the model has
#           learned 3 distinct "market personalities" from the data.
 
hmm_probs = hmm_model.predict_proba(nifty_values)
# predict_proba → for every single day in history, return a vector of
#                  3 probabilities: [P(state0), P(state1), P(state2)].
#                  These 3 values always sum to 1.0.
#                  Shape: (1762, 3)
 
hmm_df = pd.DataFrame(
    hmm_probs,
    index=nifty_ret.index,
    columns=["hmm_state_0", "hmm_state_1", "hmm_state_2"]
)
# We convert the numpy array into a DataFrame indexed by DATE.
# Why? So later we can JOIN it to stock features by matching dates.
# Each row says: "On this date, the market was X% likely to be in Regime 0,
# Y% in Regime 1, Z% in Regime 2."
 
print(f"HMM trained. {N_HMM_STATES} market regimes detected.\n")

STEP 2: TRAINING HIDDEN MARKOV MODEL


[*********************100%***********************]  1 of 1 completed


HMM trained. 3 market regimes detected.



In [7]:
# ── BLOCK 6: FEATURE ENGINEERING FUNCTION ────────────────────────────────────
 
def compute_features(ret: pd.Series) -> pd.DataFrame:
    """
    Given a Series of daily returns for ONE stock,
    compute all 7 stock-level features.
    Returns a DataFrame with one row per trading day.
    """
 
    df = pd.DataFrame(index=ret.index)
    # Start with an empty DataFrame using the same date index as the returns.
    # We'll add columns to it one by one.
 
    # ── Feature A: Downside Deviation ─────────────────────────
    neg = ret.copy()
    # Make a copy so we don't modify the original return series.
 
    neg[neg > 0] = 0
    # Zero out all POSITIVE daily returns.
    # We keep only the negative return days intact.
    # Example: [+0.02, -0.03, +0.01, -0.05] → [0, -0.03, 0, -0.05]
 
    df["downside_dev_20"] = neg.rolling(20).std()
    # rolling(20) → for each day, look at the PREVIOUS 20 days.
    # .std()      → compute standard deviation of those 20 (zeroed) returns.
    # A high value means: when this stock falls, it falls HARD.
    # This is the single most powerful crash predictor in our feature set.
 
    # ── Feature B: Kurtosis ────────────────────────────────────
    df["kurtosis_20"] = ret.rolling(20).kurt()
    # kurt()  → measures how "fat" the tails of the 20-day return distribution are.
    # Normal distribution kurtosis = 3.
    # kurtosis > 3 → more extreme moves than expected → crash risk HIGHER.
    # Think of it as: "how often does this stock make surprisingly large moves?"
 
    # ── Feature C: Skewness ────────────────────────────────────
    df["skewness_20"] = ret.rolling(20).skew()
    # skew()  → measures asymmetry of the 20-day return distribution.
    # Negative skew → small gains frequently, but large losses occasionally.
    # This is the classic "crash pattern" — the stock lulls you, then punishes you.
 
    # ── Feature D: Momentum ────────────────────────────────────
    df["momentum_5d"] = ret.rolling(5).sum()
    # Sum of last 5 days' returns.
    # If momentum_5d is already deeply negative, the stock is already falling.
    # Crashes often START before we label them — this captures early warning.
 
    # ── Feature E: Volatility & Spike Ratio ───────────────────
    df["vol_10d"]  = ret.rolling(10).std()
    df["vol_20d"]  = ret.rolling(20).std()
    # Two volatility windows: short (10 days) and medium (20 days).
 
    df["vol_ratio"] = df["vol_10d"] / (df["vol_20d"] + 1e-8)
    # vol_ratio → short-term vol DIVIDED BY medium-term vol.
    # If vol_ratio > 1.0, volatility is SPIKING (short-term > long-term).
    # A sudden vol spike is one of the classic early warning signs of a crash.
    # We add 1e-8 (a tiny number) to the denominator to prevent division by zero.
 
    return df

In [8]:
# ── BLOCK 7: CRASH LABEL FUNCTION ─────────────────────────────────────────────
 
def compute_crash_label(price_series: pd.Series, window: int = CRASH_WINDOW) -> pd.Series:
    """
    For each day t, look at the NEXT `window` trading days.
    Compute the drawdown over that forward window.
    Return 1 if drawdown > CRASH_THRESHOLD, else 0.
    """
 
    labels = pd.Series(index=price_series.index, dtype=float)
    # Start with an empty Series of NaN values, same date index as prices.
 
    for i in range(len(price_series) - window):
        # Loop over every day EXCEPT the last `window` days.
        # Why stop early? The last 10 days have no "10-day future" to look at.
 
        future = price_series.iloc[i+1 : i+1+window].values
        # iloc[i+1 : i+1+10] → the NEXT 10 days of prices.
        # i+1 means "start from tomorrow" (not today — that would be cheating!).
        # .values → converts to a numpy array for fast computation.
 
        if len(future) == 0:
            continue
        # Safety check: skip if the slice is somehow empty.
 
        entry_price = future[0]
        # The price we "enter" at tomorrow (the first day of the forward window).
 
        lowest_price = future.min()
        # The worst price reached during the 10-day window.
 
        drawdown = (entry_price - lowest_price) / (entry_price + 1e-8)
        # drawdown formula: how far did the price fall from the entry price?
        # Example: entry=100, lowest=88 → drawdown = (100-88)/100 = 0.12 = 12%
 
        labels.iloc[i] = 1.0 if drawdown > CRASH_THRESHOLD else 0.0
        # If the stock fell more than 10% → label this day as a CRASH WARNING (1).
        # Otherwise → label it SAFE (0).
 
    return labels

In [9]:
# ── BLOCK 8: BUILD THE PANEL DATASET ─────────────────────────────────────────
 
print("=" * 60)
print("STEP 3: BUILDING PANEL DATASET")
print("=" * 60)
 
all_panels = []
# We'll collect one DataFrame per stock, then stack them all at the end.
# This "long format" panel has rows = (stock × date) combinations.
 
for ticker in TICKERS:
 
    if ticker not in prices.columns:
        continue
    # Safety check: skip if this ticker somehow didn't download.
 
    price_col  = prices[ticker].ffill()
    # ffill() → "forward fill." If a stock's price is missing on a day
    #            (e.g., trading halt), carry forward the last known price.
    #            This prevents NaN from spreading into our calculations.
 
    return_col = returns[ticker].fillna(0)
    # Fill any remaining missing returns with 0 (neutral/flat day).
 
    feat_df = compute_features(return_col)
    # Call our function from Block 6 to compute all 7 features for this stock.
 
    feat_df["ticker"] = ticker
    # Add a column recording WHICH stock this row belongs to.
    # Important for identifying stocks in the final output.
 
    feat_df["date"] = feat_df.index
    # Also store the date as a regular column (not just the index).
    # This makes it easy to filter by date later (e.g., train/test split).
 
    feat_df["crash_label"] = compute_crash_label(price_col)
    # Call Block 7 to compute the binary crash target for this stock.
    # This adds the 1/0 labels alongside the features.
 
    feat_df = feat_df.join(hmm_df, how="left")
    # .join() → merges two DataFrames by their INDEX (which is the date).
    # "left" → keep all rows from feat_df; bring in HMM columns where dates match.
    # This adds the 3 HMM regime probability columns to our stock features.
 
    feat_df[["hmm_state_0","hmm_state_1","hmm_state_2"]] = \
        feat_df[["hmm_state_0","hmm_state_1","hmm_state_2"]].fillna(1/3)
    # For any dates where HMM data is missing (e.g., market holidays),
    # fill with 1/3 each — representing complete uncertainty across 3 states.
    # 1/3 ≈ 0.333 means "equally likely to be in any regime."
 
    all_panels.append(feat_df)
    # Add this stock's feature DataFrame to our list.
 
panel = pd.concat(all_panels, axis=0).dropna().reset_index(drop=True)
# pd.concat(..., axis=0) → stack all 10 per-stock DataFrames vertically.
#                           The result has (1762 days × 10 stocks) = ~17,620 rows.
# .dropna()              → remove rows where ANY feature is NaN.
#                           The first ~20 rows per stock are always NaN because
#                           rolling windows need at least 20 days of history.
# .reset_index(drop=True)→ renumber rows 0, 1, 2... instead of keeping old indices.
 
print(f"Panel shape  : {panel.shape}")
print(f"Crash events : {int(panel['crash_label'].sum())} ({panel['crash_label'].mean()*100:.1f}%)")
print(f"Safe events  : {int((panel['crash_label']==0).sum())}\n")
# Add this check at the end of Block 8
print(f"Panel shape: {panel.shape}")
if panel.shape[0] == 0:
    print("ERROR: panel is empty — check Block 7 and Block 8")
    

STEP 3: BUILDING PANEL DATASET
Panel shape  : (41590, 13)
Crash events : 1841 (4.4%)
Safe events  : 39749

Panel shape: (41590, 13)


In [10]:
FEATURE_COLS = [
    "downside_dev_20", "kurtosis_20",  "skewness_20",
    "momentum_5d",     "vol_10d",      "vol_20d",
    "vol_ratio",       "hmm_state_0",  "hmm_state_1",
    "hmm_state_2"
]
print("FEATURE_COLS defined:", FEATURE_COLS)

FEATURE_COLS defined: ['downside_dev_20', 'kurtosis_20', 'skewness_20', 'momentum_5d', 'vol_10d', 'vol_20d', 'vol_ratio', 'hmm_state_0', 'hmm_state_1', 'hmm_state_2']


In [11]:
# ── BLOCK 9: WALK-FORWARD TRAIN / TEST SPLIT ──────────────────────────────────
 
print("=" * 60)
print("STEP 4: WALK-FORWARD TRAIN/TEST SPLIT")
print("=" * 60)
 
train_df = panel[panel["date"] <= TRAIN_CUTOFF]
# Select all rows where the date is ON OR BEFORE June 30, 2023.
# This is our TRAINING set — everything the model is allowed to learn from.
 
test_df  = panel[panel["date"] >= TEST_START]
# Select all rows FROM July 1, 2023 onwards.
# This is our TEST set — data the model has NEVER seen during training.
# Key rule: test data must be entirely AFTER training data. No overlap.
 
X_train = train_df[FEATURE_COLS].values
# Extract only the feature columns. .values converts to a numpy 2D array.
# Shape: (n_train_rows, 10 features)
 
y_train = train_df["crash_label"].values.astype(int)
# Extract the labels. .astype(int) ensures 0/1 integers (not floats).
 
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df["crash_label"].values.astype(int)
 
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
# fit_transform → TWO steps in one:
#   1. fit: LEARN the mean and std of each feature FROM TRAINING DATA.
#   2. transform: APPLY the scaling to training data.
# We do BOTH on training data.
 
X_test  = scaler.transform(X_test)
# transform ONLY — we apply the TRAINING stats to the test data.
# We do NOT refit on test data. Why? Because that would be "leaking"
# information about the future into our data pipeline.
 
print(f"Training rows : {len(X_train)}")
print(f"Test rows     : {len(X_test)}\n")

STEP 4: WALK-FORWARD TRAIN/TEST SPLIT
Training rows : 35540
Test rows     : 6050



In [12]:
# ── BLOCK 10: CLASS IMBALANCE ─────────────────────────────────────────────────
 
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
spw   = round(n_neg / (n_pos + 1e-8), 2)
# scale_pos_weight (spw) = count(safe days) / count(crash days)
# If crashes are 10% of data: spw = 9000 / 1000 = 9.0
# This tells XGBoost: "treat each crash sample as if it's 9 safe samples."
# Without this, the model learns to just predict "safe" every day — which
# is 90% accurate but completely useless for our purpose.
 
print(f"Safe days  (class 0): {n_neg}")
print(f"Crash days (class 1): {n_pos}")
print(f"scale_pos_weight    : {spw}\n")

Safe days  (class 0): 33737
Crash days (class 1): 1803
scale_pos_weight    : 18.71



In [13]:
# ── BLOCK 11: OPTUNA WITH CROSS-VALIDATION ────────────────────
print("=" * 60)
print("STEP 5: OPTUNA HYPERPARAMETER SEARCH")
print("=" * 60)

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
import numpy as np

def optuna_objective(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 300),
        "max_depth":        trial.suggest_int("max_depth", 3, 5),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "min_child_weight": trial.suggest_int("min_child_weight", 20, 80),
        "reg_lambda":       trial.suggest_float("reg_lambda", 0.5, 3.0),
        "scale_pos_weight": spw,
        "eval_metric":      "logloss",
        "use_label_encoder": False,
        "random_state":     42,
        "verbosity":        0
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_train)[:, 1]
    return roc_auc_score(y_train, preds)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize")
study.optimize(optuna_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_params = study.best_params
best_params.update({
    "scale_pos_weight": spw,
    "eval_metric":      "logloss",
    "use_label_encoder": False,
    "random_state":     42,
    "verbosity":        0
})
print(f"Best params: {best_params}")

STEP 5: OPTUNA HYPERPARAMETER SEARCH


Best trial: 12. Best value: 0.955007: 100%|███████████████████████| 30/30 [00:08<00:00,  3.64it/s]

Best params: {'n_estimators': 267, 'max_depth': 5, 'learning_rate': 0.09722873355774211, 'subsample': 0.8061171777536863, 'colsample_bytree': 0.6057131122423857, 'min_child_weight': 64, 'reg_lambda': 2.244090169800752, 'scale_pos_weight': 18.71, 'eval_metric': 'logloss', 'use_label_encoder': False, 'random_state': 42, 'verbosity': 0}


In [14]:
# ── BLOCK 12: TRAIN FINAL MODEL ───────────────────────────────────────────────
 
print("=" * 60)
print("STEP 6: TRAINING FINAL MODEL C")
print("=" * 60)

model_c = xgb.XGBClassifier(**best_params)
model_c.fit(X_train, y_train)
print("Model trained.")

STEP 6: TRAINING FINAL MODEL C
Model trained.


In [15]:
# ── SAVE MODEL OBJECTS FOR LIVE SCORING ───────────────────────
import pickle
import os

# Create a folder to keep things organised
save_folder = "model_c_files"
os.makedirs(save_folder, exist_ok=True)

# Save all three objects
with open(f"{save_folder}/model_c.pkl",   "wb") as f: pickle.dump(model_c,   f)
with open(f"{save_folder}/scaler.pkl",    "wb") as f: pickle.dump(scaler,    f)
with open(f"{save_folder}/hmm_model.pkl", "wb") as f: pickle.dump(hmm_model, f)

# Verify they saved correctly
print("Files saved to folder:", save_folder)
print()
for filename in ["model_c.pkl", "scaler.pkl", "hmm_model.pkl"]:
    path     = f"{save_folder}/{filename}"
    size_kb  = os.path.getsize(path) / 1024
    print(f"  ✅ {filename:20s}  {size_kb:.1f} KB")

print()
print("Done. These 3 files are all you need to run live_scores.py")


Files saved to folder: model_c_files

  ✅ model_c.pkl           490.3 KB
  ✅ scaler.pkl            0.6 KB
  ✅ hmm_model.pkl         2.7 KB

Done. These 3 files are all you need to run live_scores.py


In [16]:
# ── BLOCK 13: EVALUATE ON TEST SET ────────────────────────────────────────────
 
print("=" * 60)
print("STEP 7: EVALUATION ON OUT-OF-SAMPLE TEST SET")
print("=" * 60)
 
y_pred_proba = model_c.predict_proba(X_test)[:, 1]
# Ask the model: "for each row in the test set, what is the probability
# of a crash?" Returns a value between 0.0 and 1.0 for each row.
 
y_pred_class = (y_pred_proba >= 0.08).astype(int)
# Convert probabilities to hard decisions.
# If P(crash) >= 50% → predict CRASH (1). Otherwise → predict SAFE (0).
# You can LOWER this threshold (e.g., 0.35) to catch more crashes
# at the cost of more false alarms.
 
print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_class, target_names=["No Crash", "Crash"]))
# Prints: Precision, Recall, F1-Score, Support for both classes.
# Focus on the "Crash" row — that's our model's performance on what matters.
 
cm = confusion_matrix(y_test, y_pred_class)
# cm is a 2×2 matrix:
#        Predicted: 0   Predicted: 1
# True 0: [True Neg,    False Pos]
# True 1: [False Neg,   True Pos ]
 
print("CONFUSION MATRIX:")
print(f"  True Negatives  (safe days correctly identified) : {cm[0,0]}")
print(f"  False Positives (false alarms — safe but flagged): {cm[0,1]}")
print(f"  False Negatives (missed crashes — DANGEROUS)     : {cm[1,0]}")
print(f"  True Positives  (crashes correctly caught)       : {cm[1,1]}")
 
auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nAUC-ROC Score : {auc:.4f}  (target: >0.65)\n")


STEP 7: EVALUATION ON OUT-OF-SAMPLE TEST SET

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    No Crash       1.00      0.59      0.74      6012
       Crash       0.01      0.68      0.02        38

    accuracy                           0.59      6050
   macro avg       0.50      0.64      0.38      6050
weighted avg       0.99      0.59      0.74      6050

CONFUSION MATRIX:
  True Negatives  (safe days correctly identified) : 3541
  False Positives (false alarms — safe but flagged): 2471
  False Negatives (missed crashes — DANGEROUS)     : 12
  True Positives  (crashes correctly caught)       : 26

AUC-ROC Score : 0.6834  (target: >0.65)



In [17]:
# ── BLOCK 14: FEATURE IMPORTANCE ──────────────────────────────────────────────
 
print("=" * 60)
print("STEP 8: FEATURE IMPORTANCE")
print("=" * 60)
print("(Which inputs matter most for detecting crashes?)\n")
 
imp_df = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "importance": model_c.feature_importances_
# feature_importances_ → XGBoost records how much each feature
#                        contributed to reducing prediction error across all trees.
#                        Higher = more useful for crash detection.
}).sort_values("importance", ascending=False)
# Sort so the most important feature appears first.
 
for _, row in imp_df.iterrows():
    bar_len = int(row["importance"] / imp_df["importance"].max() * 30)
    bar     = "█" * bar_len
    # Visual bar chart in terminal: scale each bar relative to the max importance.
    print(f"  {row['feature']:20s}  {bar:<30s}  {row['importance']:.4f}")
 
print()

STEP 8: FEATURE IMPORTANCE
(Which inputs matter most for detecting crashes?)

  hmm_state_2           ██████████████████████████████  0.2652
  hmm_state_0           █████████████                   0.1156
  vol_20d               ██████████                      0.0904
  downside_dev_20       █████████                       0.0829
  vol_10d               █████████                       0.0822
  skewness_20           ████████                        0.0786
  hmm_state_1           ████████                        0.0780
  kurtosis_20           ████████                        0.0721
  vol_ratio             ███████                         0.0698
  momentum_5d           ███████                         0.0652



In [18]:
# ── BLOCK 15: LIVE CRASH RISK SCORES ─────────────────────────────────────────
 
print("=" * 60)
print("STEP 9: LIVE CRASH RISK SCORES (Latest Data)")
print("=" * 60)
print("(What is each stock's crash probability RIGHT NOW?)\n")
 
live_scores = []
 
for ticker in TICKERS:
    if ticker not in prices.columns:
        continue
 
    feat = compute_features(returns[ticker].fillna(0))
    # Recompute features for this ticker using the full price history.
 
    feat = feat.join(hmm_df, how="left")
    feat[["hmm_state_0","hmm_state_1","hmm_state_2"]] = \
        feat[["hmm_state_0","hmm_state_1","hmm_state_2"]].fillna(1/3)
    # Add the HMM regime probabilities, same as in the training pipeline.
 
    latest_row = feat[FEATURE_COLS].dropna().iloc[[-1]]
    # .dropna()    → remove rows where features couldn't be computed yet.
    # .iloc[[-1]]  → grab only the LAST row (most recent trading day).
    #                The double brackets [[-1]] keep it as a DataFrame (not a Series).
 
    if latest_row.empty:
        continue
    # If for some reason we couldn't compute features, skip this ticker.
 
    latest_scaled = scaler.transform(latest_row)
    # Apply the SAME scaling transformation used on training data.
    # Critical: we must transform the live data identically to how we
    # transformed the training data, or the model sees completely different numbers.
 
    crash_prob = model_c.predict_proba(latest_scaled)[0, 1]
    # Get the crash probability for this single row.
    # [0, 1] → first (only) row, second column (P(crash)).
 
    if   crash_prob > 0.60: risk_level = "🔴 HIGH RISK   — Consider avoiding / hedging"
    elif crash_prob > 0.20: risk_level = "🟡 MEDIUM RISK — Monitor closely"
    else:                   risk_level = "🟢 LOW RISK    — Relatively safe"
 
    live_scores.append({
        "Ticker"      : ticker,
        "Crash Prob %" : f"{crash_prob * 100:.1f}%",
        "Risk Level"  : risk_level
    })
 
scores_df = pd.DataFrame(live_scores).sort_values("Crash Prob %", ascending=False)
print(scores_df.to_string(index=False))
 
print("\n" + "=" * 60)
print("MODEL C COMPLETE.")
print("Use the 'Crash Prob %' column to FILTER OUT high-risk stocks")
print("before passing the list to your portfolio construction layer.")
print("=" * 60)

STEP 9: LIVE CRASH RISK SCORES (Latest Data)
(What is each stock's crash probability RIGHT NOW?)

       Ticker Crash Prob %                      Risk Level
  HDFCBANK.NS         8.2% 🟢 LOW RISK    — Relatively safe
       TCS.NS         7.1% 🟢 LOW RISK    — Relatively safe
      INFY.NS         5.3% 🟢 LOW RISK    — Relatively safe
      SBIN.NS         4.9% 🟢 LOW RISK    — Relatively safe
 KOTAKBANK.NS         3.1% 🟢 LOW RISK    — Relatively safe
       ITC.NS        28.6% 🟡 MEDIUM RISK — Monitor closely
BHARTIARTL.NS        22.4% 🟡 MEDIUM RISK — Monitor closely
HINDUNILVR.NS         2.9% 🟢 LOW RISK    — Relatively safe
  RELIANCE.NS        18.3% 🟢 LOW RISK    — Relatively safe
 ICICIBANK.NS        16.0% 🟢 LOW RISK    — Relatively safe

MODEL C COMPLETE.
Use the 'Crash Prob %' column to FILTER OUT high-risk stocks
before passing the list to your portfolio construction layer.


In [19]:
# ── BLOCK 15: LIVE CRASH RISK SCORES ─────────────────────────────────────────
 
print("=" * 60)
print("STEP 9: LIVE CRASH RISK SCORES (Latest Data)")
print("=" * 60)
print("(What is each stock's crash probability RIGHT NOW?)\n")
 
live_scores = []
 
for ticker in TICKERS:
    if ticker not in prices.columns:
        continue
 
    feat = compute_features(returns[ticker].fillna(0))
    # Recompute features for this ticker using the full price history.
 
    feat = feat.join(hmm_df, how="left")
    feat[["hmm_state_0","hmm_state_1","hmm_state_2"]] = \
        feat[["hmm_state_0","hmm_state_1","hmm_state_2"]].fillna(1/3)
    # Add the HMM regime probabilities, same as in the training pipeline.
 
    latest_row = feat[FEATURE_COLS].dropna().iloc[[-1]]
    # .dropna()    → remove rows where features couldn't be computed yet.
    # .iloc[[-1]]  → grab only the LAST row (most recent trading day).
    #                The double brackets [[-1]] keep it as a DataFrame (not a Series).
 
    if latest_row.empty:
        continue
    # If for some reason we couldn't compute features, skip this ticker.
 
    latest_scaled = scaler.transform(latest_row)
    # Apply the SAME scaling transformation used on training data.
    # Critical: we must transform the live data identically to how we
    # transformed the training data, or the model sees completely different numbers.
 
    crash_prob = model_c.predict_proba(latest_scaled)[0, 1]
    # Get the crash probability for this single row.
    # [0, 1] → first (only) row, second column (P(crash)).
 
    if   crash_prob > 0.40: risk_level = "🔴 HIGH RISK   — Consider avoiding / hedging"
    elif crash_prob > 0.20: risk_level = "🟡 MEDIUM RISK — Monitor closely"
    else:                   risk_level = "🟢 LOW RISK    — Relatively safe"
 
    live_scores.append({
        "Ticker"      : ticker,
        "Crash Prob %" : f"{crash_prob * 100:.1f}%",
        "Risk Level"  : risk_level
    })
 
scores_df = pd.DataFrame(live_scores).sort_values("Crash Prob %", ascending=False)
print(scores_df.to_string(index=False))
 
print("\n" + "=" * 60)
print("MODEL C COMPLETE.")
print("Use the 'Crash Prob %' column to FILTER OUT high-risk stocks")
print("before passing the list to your portfolio construction layer.")
print("=" * 60)

STEP 9: LIVE CRASH RISK SCORES (Latest Data)
(What is each stock's crash probability RIGHT NOW?)

       Ticker Crash Prob %                      Risk Level
  HDFCBANK.NS         8.2% 🟢 LOW RISK    — Relatively safe
       TCS.NS         7.1% 🟢 LOW RISK    — Relatively safe
      INFY.NS         5.3% 🟢 LOW RISK    — Relatively safe
      SBIN.NS         4.9% 🟢 LOW RISK    — Relatively safe
 KOTAKBANK.NS         3.1% 🟢 LOW RISK    — Relatively safe
       ITC.NS        28.6% 🟡 MEDIUM RISK — Monitor closely
BHARTIARTL.NS        22.4% 🟡 MEDIUM RISK — Monitor closely
HINDUNILVR.NS         2.9% 🟢 LOW RISK    — Relatively safe
  RELIANCE.NS        18.3% 🟢 LOW RISK    — Relatively safe
 ICICIBANK.NS        16.0% 🟢 LOW RISK    — Relatively safe

MODEL C COMPLETE.
Use the 'Crash Prob %' column to FILTER OUT high-risk stocks
before passing the list to your portfolio construction layer.


In [20]:
import os
print(os.getcwd())


/Users/mohitbajaj007/Desktop/proj


In [21]:
# ── GENERATE CSV FROM JUPYTER — no Terminal needed ────────────
import pandas as pd
import os

scores = []
for ticker in TICKERS:
    if ticker not in prices.columns:
        continue
    feat = compute_features(returns[ticker].fillna(0))
    feat = feat.join(hmm_df, how="left")
    feat[["hmm_state_0","hmm_state_1","hmm_state_2"]] = \
        feat[["hmm_state_0","hmm_state_1","hmm_state_2"]].fillna(1/3)

    latest = feat[FEATURE_COLS].dropna().iloc[[-1]]
    if latest.empty:
        continue

    prob  = model_c.predict_proba(scaler.transform(latest))[0, 1]
    bear  = hmm_df.iloc[-1]["hmm_state_2"]
    side  = hmm_df.iloc[-1]["hmm_state_1"]
    bull  = hmm_df.iloc[-1]["hmm_state_0"]

    if   prob > 0.40: risk = "HIGH RISK   — Avoid / Exit"
    elif prob > 0.20: risk = "MEDIUM RISK — Reduce size"
    else:             risk = "LOW RISK    — Safe to hold"

    scores.append({
        "Ticker"            : ticker,
        "Crash Probability" : round(prob * 100, 2),
        "Risk Level"        : risk,
        "Bear Regime %"     : round(bear * 100, 2),
        "Sideways Regime %"  : round(side * 100, 2),
        "Bull Regime %"     : round(bull * 100, 2),
        "Date"              : pd.Timestamp.today().strftime("%Y-%m-%d"),
        "Time"              : pd.Timestamp.today().strftime("%H:%M"),
    })

# Create DataFrame and sort by crash probability
df_live = pd.DataFrame(scores).sort_values("Crash Probability", ascending=False)

# Save to CSV
csv_path = os.path.join(os.getcwd(), "live_crash_scores.csv")
df_live.to_csv(csv_path, index=False)

# Display in notebook
print("=" * 65)
print(f"MODEL C — LIVE CRASH RISK SCORES")
print("=" * 65)
print(df_live.to_string(index=False))
print()
print(f"✅ CSV saved to: {csv_path}")

MODEL C — LIVE CRASH RISK SCORES
       Ticker  Crash Probability                 Risk Level  Bear Regime %  Sideways Regime %  Bull Regime %       Date  Time
       ITC.NS          28.620001  MEDIUM RISK — Reduce size           0.45              57.72          41.83 2026-03-22 23:19
BHARTIARTL.NS          22.370001  MEDIUM RISK — Reduce size           0.45              57.72          41.83 2026-03-22 23:19
  RELIANCE.NS          18.299999 LOW RISK    — Safe to hold           0.45              57.72          41.83 2026-03-22 23:19
 ICICIBANK.NS          16.010000 LOW RISK    — Safe to hold           0.45              57.72          41.83 2026-03-22 23:19
  HDFCBANK.NS           8.170000 LOW RISK    — Safe to hold           0.45              57.72          41.83 2026-03-22 23:19
       TCS.NS           7.060000 LOW RISK    — Safe to hold           0.45              57.72          41.83 2026-03-22 23:19
      INFY.NS           5.270000 LOW RISK    — Safe to hold           0.45           

In [1]:
import os
print(os.getcwd())

/Users/mohitbajaj007/Desktop/proj
